<a href="https://colab.research.google.com/github/Ech012/Nitzanim-Hackathon-ML-2026/blob/main/hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers[torch] datasets accelerate catboost openpyxl scikit-learn pandas numpy


In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from catboost import CatBoostClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from transformers import EarlyStoppingCallback

In [ ]:
# 1. SETUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
file_name = 'Cyberbullying Dataset.xlsx'

df = pd.read_excel(file_name)
df['tweet_text'] = df['tweet_text'].astype(str).fillna('')

# 2. FEATURE ENGINEERING (Criteria 2a)
def get_feats(text):
    return pd.Series({
        'shout': sum(1 for c in text if c.isupper()) / (len(text) + 1),
        'punc': text.count('!') + text.count('?'),
        'len': len(text)
    })

manual_feats = df['tweet_text'].apply(get_feats)
df = pd.concat([df, manual_feats], axis=1)

labels = df['cyberbullying_type'].unique().tolist()
label2id = {l: i for i, l in enumerate(labels)}
df['label'] = df['cyberbullying_type'].map(label2id)

train_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])

# 3. PHASE 1: ROBERTA (Stability & Accuracy)
MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def tok_fn(batch):
    return tokenizer(batch["tweet_text"], truncation=True, max_length=128, padding="max_length")

train_ds = Dataset.from_pandas(train_df[['tweet_text', 'label']].reset_index(drop=True)).map(tok_fn, batched=True)
test_ds = Dataset.from_pandas(test_df[['tweet_text', 'label']].reset_index(drop=True)).map(tok_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels), ignore_mismatched_sizes=True).to(device)

# הגדרות הכי בסיסיות שיש למניעת TypeErrors
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
    fp16=True,
    logging_steps=100,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    processing_class=tokenizer
)

print("\n🧠 Training Phase 1...")
trainer.train()

# 4. PHASE 2: CATBOOST STACKING (The 85% Secret)
print("\n🚀 Phase 2: Stacking...")
train_probs = trainer.predict(train_ds).predictions
test_probs = trainer.predict(test_ds).predictions

X_train = np.hstack([train_probs, train_df[['shout', 'punc', 'len']].values])
X_test = np.hstack([test_probs, test_df[['shout', 'punc', 'len']].values])

cb = CatBoostClassifier(iterations=500, depth=6, verbose=0, task_type="GPU")
cb.fit(X_train, train_df['label'])

# 5. RESULTS
y_pred = cb.predict(X_test)
print(classification_report(test_df['label'], y_pred, target_names=labels))

In [ ]:
import joblib
import shutil
from google.colab import files

# 1. שמירת ה-Transformer (החלק של RoBERTa שמאומן על הטקסט)
trainer.save_model("./final_cyber_model")
tokenizer.save_pretrained("./final_cyber_model")

# 2. שמירת ה-CatBoost (השכבה ההיברידית שסוגרת את הדיוק ל-86%)
joblib.dump(cb, 'hybrid_classifier.joblib')

# 3. כיווץ התיקייה לקובץ ZIP אחד כדי שיהיה קל להוריד
shutil.make_archive('cyber_model_package', 'zip', './final_cyber_model')

print("✅ המודלים נשמרו בזיכרון הזמני של הקולאב!")

In [ ]:
from google.colab import files
files.download('cyber_model_package.zip')
files.download('hybrid_classifier.joblib')